# 🎬 06 — Video Inference
## SmartMine Vision AI · Stage 1: PPE Detection

---

### Objectives
1. Run the full **PPE detection + compliance pipeline** on a video.
2. Overlay **bounding boxes, class labels, confidence, and FPS counter**.
3. Apply **SAFE/UNSAFE banner** per worker per frame.
4. Measure **real-time performance** (FPS, latency).
5. Save annotated video to `outputs/videos/`.

---

### Pipeline (per frame)

```
read frame
    → YOLOv8 predict (conf ≥ 0.40)
    → draw_detections (boxes + labels)
    → classify_workers (IoU-based SAFE/UNSAFE)
    → overlay compliance banners
    → FPS counter overlay
    → write to output video
```

> **How to get a test video:**
> Place any `.mp4` / `.avi` construction or mining site video in the project
> folder and update `VIDEO_PATH` in the Setup cell below.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Resolve project root by walking up until 'src/' is found
_cwd = Path().resolve()
PROJECT_ROOT = _cwd
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.ppe_detection.utils import MODELS_DIR, OUTPUTS_DIR, ensure_dirs
from src.ppe_detection.inference import load_model, predict_image, draw_detections
from src.ppe_detection.ppe_classifier import classify_workers, compliance_color, ComplianceStatus

ensure_dirs()

WEIGHTS = MODELS_DIR / "yolov8n_smartmine_baseline.pt"

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
VIDEO_PATH = Path("/path/to/your/construction_video.mp4")
# ─────────────────────────────────────────────────────────────────────────────

if not WEIGHTS.exists():
    print("⚠  Model not found — run notebook 03 first.")
else:
    model = load_model(WEIGHTS)
    print(f"Model loaded : {WEIGHTS.name}")

print(f"Video path   : {VIDEO_PATH}")
print(f"Video exists : {VIDEO_PATH.exists()}")

## 2. Video Properties Inspection

In [ ]:
if VIDEO_PATH.exists():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    props = {
        "Width"        : int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "Height"       : int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "FPS"          : round(cap.get(cv2.CAP_PROP_FPS), 2),
        "Frame count"  : int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "Duration (s)" : round(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) /
                          max(cap.get(cv2.CAP_PROP_FPS), 1), 1),
    }
    cap.release()

    print("VIDEO PROPERTIES")
    print("=" * 35)
    for k, v in props.items():
        print(f"  {k:<18}: {v}")
    print("=" * 35)
else:
    print("Set VIDEO_PATH above to a valid video file.")

## 3. Full Inference Pipeline

In [ ]:
def run_pipeline(video_path: Path, conf: float = 0.40) -> Path:
    """Run detection + compliance on every frame. Returns output path."""
    cap   = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_path = OUTPUTS_DIR / "videos" / f"ppe_{video_path.stem}.mp4"
    writer   = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    frame_times: list[float] = []
    stats = {"frames": 0, "total_dets": 0, "safe": 0, "unsafe": 0}

    while cap.isOpened():
        t0   = time.perf_counter()
        ret, frame = cap.read()
        if not ret:
            break

        dets    = predict_image(model, frame, conf=conf)
        canvas  = draw_detections(frame, dets)
        workers = classify_workers(dets)

        # Compliance overlay
        for w_obj in workers:
            x1, y1, x2, y2 = w_obj.person_bbox
            color = compliance_color(w_obj.status)
            cv2.rectangle(canvas, (x1, y1 - 28), (x2, y1 - 2), color, -1)
            cv2.putText(canvas, w_obj.status.value, (x1 + 5, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.72, (0, 0, 0), 2)
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 3)

        # FPS counter
        elapsed = time.perf_counter() - t0
        frame_times.append(elapsed)
        if len(frame_times) > 30:
            frame_times.pop(0)
        live_fps = 1.0 / (sum(frame_times) / len(frame_times))

        safe_n   = sum(1 for ww in workers if ww.status == ComplianceStatus.SAFE)
        unsafe_n = len(workers) - safe_n

        # HUD overlay
        hud_lines = [
            f"FPS: {live_fps:.1f}",
            f"Frame: {stats['frames']+1}/{total}",
            f"Detections: {len(dets)}",
            f"SAFE: {safe_n}  UNSAFE: {unsafe_n}",
        ]
        for j, line in enumerate(hud_lines):
            cv2.putText(canvas, line, (10, 30 + j * 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75,
                        (255, 255, 0) if j == 0 else (200, 200, 200), 2)

        writer.write(canvas)
        stats["frames"]     += 1
        stats["total_dets"] += len(dets)
        stats["safe"]       += safe_n
        stats["unsafe"]     += unsafe_n

        if stats["frames"] % 50 == 0:
            print(f"  Frame {stats['frames']:4d}/{total} | FPS {live_fps:.1f} | "
                  f"SAFE {safe_n} UNSAFE {unsafe_n}")

    cap.release()
    writer.release()
    return out_path, stats

if VIDEO_PATH.exists() and WEIGHTS.exists():
    out_path, run_stats = run_pipeline(VIDEO_PATH, conf=0.40)
    print(f"\n✅ Output → {out_path}")
else:
    print("Update VIDEO_PATH and ensure model exists.")

## 4. Run Statistics

In [ ]:
if VIDEO_PATH.exists() and WEIGHTS.exists():
    total_w = run_stats["safe"] + run_stats["unsafe"]
    print("RUN STATISTICS")
    print("=" * 45)
    print(f"  Frames processed  : {run_stats['frames']}")
    print(f"  Total detections  : {run_stats['total_dets']}")
    print(f"  Avg dets/frame    : {run_stats['total_dets']/max(run_stats['frames'],1):.1f}")
    print(f"  SAFE detections   : {run_stats['safe']}")
    print(f"  UNSAFE detections : {run_stats['unsafe']}")
    if total_w:
        pct = 100 * run_stats["safe"] / total_w
        print(f"  Compliance rate   : {pct:.1f}%")
    print("=" * 45)

## 5. Preview First & Last Frame

In [ ]:
if VIDEO_PATH.exists() and WEIGHTS.exists():
    cap = cv2.VideoCapture(str(out_path))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for i, (ax, pos) in enumerate(zip(axes, [0, run_stats["frames"] - 1])):
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ret, frame = cap.read()
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Frame {pos+1}", fontsize=11)
        ax.axis("off")

    cap.release()
    plt.suptitle("Video Inference — First & Last Frame Preview", fontsize=13)
    plt.tight_layout()
    preview_path = OUTPUTS_DIR / "images" / "video_preview.png"
    plt.savefig(str(preview_path), dpi=150)
    plt.show()
    print(f"Saved → {preview_path}")

## 6. Performance Benchmarks

| Hardware | YOLOv8n 640px | Real-time? |
|---|---|---|
| RTX 3060 | ~80–120 FPS | ✅ Yes |
| RTX 2060 | ~50–80 FPS | ✅ Yes |
| Apple M2 (MPS) | ~35–55 FPS | ✅ Yes |
| Apple M1 (MPS) | ~25–40 FPS | ✅ Marginal |
| CPU only | ~5–15 FPS | ❌ Not real-time |

> For CPU deployments: use `yolov8n.onnx` export with OpenVINO or ONNX Runtime
> for 2–3× speed improvement.

## 7. Stage 1 — Pipeline Status

**What Stage 1 delivers:**

| Component | Status |
|---|---|
| Dataset explored & validated | ✅ pipeline implemented |
| YOLOv8n fine-tuned (100 epochs) | ⏳ pending — SPEC-003 |
| Quantitative evaluation (mAP50, PR, CM) | ⏳ pending — SPEC-003 |
| Image inference + compliance overlay | ✅ pipeline implemented |
| Video inference + real-time FPS | ✅ pipeline implemented |
| SAFE/UNSAFE worker classification | ✅ pipeline implemented |
| All outputs saved to `outputs/` | ✅ pipeline implemented |

> Training and evaluation results will be populated once SPEC-003 (Model Training & Validation) is complete.

**Stage 2 — Vehicle Detection** begins next:
- Dataset: BDD100K / COCO
- Classes: truck, car, bus, motorcycle
- Same YOLOv8 pipeline, new module: `src/vehicle_detection/`